# Hybrid Movie recommendation system

## 1. Content-Based system - Sentence Transformer SBERT

In [18]:
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
import tensorflow as tf

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/movielens-20m-dataset/rating.csv
/kaggle/input/movielens-20m-dataset/link.csv
/kaggle/input/movielens-20m-dataset/genome_tags.csv
/kaggle/input/movielens-20m-dataset/genome_scores.csv
/kaggle/input/movielens-20m-dataset/tag.csv
/kaggle/input/movielens-20m-dataset/movie.csv
/kaggle/input/top-500-600-movies-of-each-year-from-1960-to-2024/final_dataset.parquet
/kaggle/input/top-500-600-movies-of-each-year-from-1960-to-2024/final_dataset.feather
/kaggle/input/top-500-600-movies-of-each-year-from-1960-to-2024/final_dataset.csv
/kaggle/input/movielens-100k/ml-latest-small/movies.csv
/kaggle/input/movielens-100k/ml-latest-small/ratings.csv
/kaggle/input/movielens-100k/ml-latest-small/README.txt
/kaggle/input/movielens-100k/ml-latest-small/tags.csv
/kaggle/input/movielens-100k/ml-latest-small/links.csv
/kaggle/input/tmdb-movie-metadata/tmdb_5000_movies.csv
/kaggle/input/tmdb-movie-metadata/tmdb_5000_credits.csv


In [1]:
movies_df2 = pd.read_csv("/kaggle/input/tmdb-movie-metadata/tmdb_5000_movies.csv")
movies_df = pd.read_csv("/kaggle/input/top-500-600-movies-of-each-year-from-1960-to-2024/final_dataset.csv")
credits_df = pd.read_csv("/kaggle/input/tmdb-movie-metadata/tmdb_5000_credits.csv")
ratings_df = pd.read_csv("/kaggle/input/movielens-100k/ml-latest-small/ratings.csv")
links_df = pd.read_csv("/kaggle/input/movielens-100k/ml-latest-small/links.csv")

links_df.columns

NameError: name 'pd' is not defined

In [31]:
movies_df.loc[movies_df['title'].str.contains("Avatar", case=False), ['id', 'title']]


,id,title
20142,tt1630029,Avatar: The Way of Water
24028,tt1757678,Avatar: Fire and Ash
42841,tt0499549,Avatar


In [21]:
# Handle missing plot overviews, as they are crucial for our model
print(f"Movies with missing overview (before): {movies_df['description'].isnull().sum()}")
movies_df['description'] = movies_df['description'].fillna('') # Replace NaN with empty string
print(f"Movies with missing overview (after): {movies_df['description'].isnull().sum()}")

print(f"Total movies in dataset: {len(movies_df)}")

Movies with missing overview (before): 2360
Movies with missing overview (after): 0
Total movies in dataset: 63249


In [22]:
movies_df.head(5)

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pan

,id,title,duration,mpa,rating,votes,méta_score,description,movie_link,writers,...,opening_weekend_gross,gross_worldwide,gross_us_canada,release_date,countries_origin,filming_locations,production_companies,awards_content,genres,languages
0,tt0027483,The Crimson Circle,1h 16m,NaN,6.4,30,NaN,An extortion ring murders anyone who refuses t...,https://www.imdb.com/title/tt0027483/,"['Reginald Denham', 'Edgar Wallace', 'Howard I...",...,NaN,NaN,NaN,1936-08-10,['United Kingdom'],NaN,['Richard Wainwright Productions'],NaN,['Drama'],['English']
1,tt0058131,The Mystery of Thug Island,1h 36m,NaN,5.0,114,NaN,"Three year old Ada, daughter of the British ca...",https://www.imdb.com/title/tt0058131/,"['Emilio Salgari', 'Arpad DeRiso', 'Ottavio Po...",...,NaN,NaN,NaN,1966-05-28,"['Italy', 'Monaco', 'West Germany']",NaN,"['Eichberg-Film', 'Liber Film']",NaN,['Adventure'],['Italian']
2,tt0042760,Las mujeres de mi general,1h 52m,Not Rated,6.8,74,NaN,Infante stars as a rebel general caught up in ...,https://www.imdb.com/title/tt0042760/,"['Joselito Rodríguez', 'Celestino Gorostiza', ...",...,NaN,NaN,NaN,1951-07-13,['Mexico'],NaN,['Producciones Rodríguez Hermanos'],NaN,"['Drama', 'War']",['Spanish']
3,tt0027667,Gentle Julia,1h 2m,Approved,6.8,38,NaN,A shy newspaperman (Brown) nearly gives up whe...,https://www.imdb.com/title/tt0027667/,"['Booth Tarkington', 'Lamar Trotti']",...,NaN,NaN,NaN,1936-04-10,['United States'],"['20th Century Fox Studios - 10201 Pico Blvd.,...",['Twentieth Century Fox'],NaN,"['Comedy', 'Drama', 'Romance']",['English']
4,tt0055747,Love at Twenty,1h 50m,NaN,7.2,2.5K,NaN,"""Love at Twenty"" unites five directors from ar...",https://www.imdb.com/title/tt0055747/,"['Shintarô Ishihara', 'Marcel Ophüls', 'Renzo ...",...,NaN,NaN,NaN,1963-02-06,"['France', 'Italy', 'Japan', 'Poland', 'West G...","['Warsaw Zoo, Ratuszowa, Praga Pólnoc, Warsaw,...","['Ulysse Productions', 'Unitec Films', 'Cinese...",NaN,"['Drama', 'Romance']","['French', 'Polish', 'Japanese', 'Italian', 'G..."


In [23]:
# Load a pre-trained SBERT model.
# 'all-MiniLM-L6-v2' is fast and provides high-quality semantic embeddings.
print("Loading Sentence-Transformer model...")
model = SentenceTransformer('all-MiniLM-L6-v2')
print("Model loaded successfully.")

Loading Sentence-Transformer model...
Model loaded successfully.


In [24]:
# Block 1: User Tower (User-Based CF)
# ------------------------------------
# This script prepares the user data and defines the
# Keras model for the User Tower. This tower's only job
# is to convert a user_id into a learned "taste" embedding.

# --- Step 0: Imports ---
import pickle
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.layers import Input, Embedding, Flatten
from tensorflow.keras.models import Model


# --- Step 2: Encode User IDs ---
# The Embedding layer needs integer indices (0, 1, 2...)
# not the raw user IDs (e.g., 1, 5, 28).

print("Encoding user IDs...")
user_encoder = LabelEncoder()

# Fit the encoder on the unique user IDs
# We use .unique() to be efficient
all_user_ids = ratings_df['userId'].unique()
user_encoder.fit(all_user_ids)

# Get the total number of unique users for the Embedding layer
n_users = len(user_encoder.classes_)
print(f"Total unique users found: {n_users}")

# --- Step 3: Save the Encoder ---
# We MUST save this encoder. We will need it later to
# transform user IDs when we build and predict with
# the final hybrid model.
with open('user_encoder.pkl', 'wb') as f:
    pickle.dump(user_encoder, f)
print("User encoder saved to 'user_encoder.pkl'")


# --- Step 4: Define the User Tower Architecture ---
# This is a Keras functional model that defines just this tower.
# It will be a "sub-model" in our final hybrid architecture.

# Embedding size is a hyperparameter. 32 or 50 is a good start.
USER_EMBEDDING_SIZE = 32 

def build_user_tower(n_users_input):
    """
    Builds the Keras model for the User Tower.
    """
    # Define the input layer
    # It expects a single integer (the user_idx)
    user_input = Input(shape=(1,), name='user_input_index')

    # Define the Embedding layer
    # This is the core of the tower. It's a lookup table that maps
    # user_idx -> user_vector
    user_embedding = Embedding(input_dim=n_users_input,
                               output_dim=USER_EMBEDDING_SIZE,
                               name='user_embedding')(user_input)
    
    # Flatten the embedding output to a 1D vector
    user_vector = Flatten(name='user_vector')(user_embedding)

    # Define the model
    model = Model(inputs=user_input, outputs=user_vector, name='User_Tower')
    return model

# --- Step 5: Build and Inspect the Tower ---
print("\nBuilding User Tower model...")
user_tower_model = build_user_tower(n_users)

# Show a summary of just this tower
user_tower_model.summary()


# --- Step 6: Test the Tower (Example) ---
# Let's see what the tower does.
print("\n--- Testing the tower ---")

# 1. Get a sample raw user ID
sample_user_id = all_user_ids[0]
print(f"Sample raw user ID: {sample_user_id}")

# 2. Transform it using our encoder
sample_user_idx = user_encoder.transform([sample_user_id])
print(f"Encoded user index: {sample_user_idx}")

# 3. Pass the index to the model
# Note: The model is not trained, so the vector will be random.
# We wrap the index in another array for batching (model expects batch)
sample_input = np.array(sample_user_idx)
sample_vector = user_tower_model.predict(sample_input)

print(f"Output vector shape: {sample_vector.shape}") # (1, 32)
print(f"Output vector (preview): {sample_vector[0, :5]}...")

Encoding user IDs...
Total unique users found: 610
User encoder saved to 'user_encoder.pkl'

Building User Tower model...


Model: "User_Tower"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ user_input_index (InputLayer)   │ (None, 1)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ user_embedding (Embedding)      │ (None, 1, 32)          │        19,520 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ user_vector (Flatten)           │ (None, 32)             │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 19,520 (76.25 KB)

 Trainable params: 19,520 (76.25 KB)

 Non-trainable params: 0 (0.00 B)


--- Testing the tower ---
Sample raw user ID: 1
Encoded user index: [0]
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 95ms/step
Output vector shape: (1, 32)
Output vector (preview): [-0.02721084  0.00143566  0.03581465 -0.02691909 -0.01408615]...


In [25]:
# Block 2: Complete Content Tower (SBERT + Features) - CORRIGÉ
# ---------------------------------------------------

# --- Step 0: Imports ---
import pandas as pd
import numpy as np
import pickle
import ast 
from sklearn.preprocessing import MultiLabelBinarizer, MinMaxScaler, LabelEncoder
from sentence_transformers import SentenceTransformer
import os

# TensorFlow / Keras Imports
import tensorflow as tf
from tensorflow.keras.layers import Input, Embedding, Dense, Flatten, Concatenate, BatchNormalization, Dropout
from tensorflow.keras.models import Model

# --- Step 1: Load Data & SBERT ---
print("Loading Data...")
# Assurez-vous que movies_df est chargé ici si ce n'est pas déjà fait
# movies_df = pd.read_csv('votre_fichier.csv') 

EMBEDDINGS_FILE = 'movie_embeddings.npy'
sbert_model = SentenceTransformer('all-MiniLM-L6-v2')

# Clean description
movies_df['description'] = movies_df['description'].fillna('')
corpus = movies_df['description'].tolist()

# Check if embeddings exist or generate them
if os.path.exists(EMBEDDINGS_FILE):
    print(f"Loading existing embeddings from {EMBEDDINGS_FILE}...")
    sbert_features = np.load(EMBEDDINGS_FILE)
else:
    print("Generating SBERT embeddings...")
    sbert_features = sbert_model.encode(corpus, show_progress_bar=True)
    np.save(EMBEDDINGS_FILE, sbert_features)

# Get SBERT dimension
SBERT_DIM = sbert_features.shape[1] # Should be 384
print(f"SBERT embedding dimension: {SBERT_DIM}")


# --- Step 3: Preprocess Other Content Features ---

print("Starting feature preprocessing...")

def parse_list_column(x):
    try:
        if isinstance(x, str):
            return ast.literal_eval(x)
        return x if isinstance(x, list) else []
    except:
        return []

# 3a. Genres
print("Processing Genres...")
movies_df['genres_list'] = movies_df['genres'].apply(parse_list_column)
mlb_genres = MultiLabelBinarizer()
genres_features = mlb_genres.fit_transform(movies_df['genres_list'])
n_genres = len(mlb_genres.classes_)

# --- DANS LE BLOC 2 ---

# 3b. Stars (Actors) - OPTIMISATION MÉMOIRE
print("Processing Stars...")
movies_df['stars_list'] = movies_df['stars'].apply(parse_list_column)

# LIMITATION IMPORTANTE : On ne garde que les 500 acteurs les plus fréquents
# Cela réduit la largeur de la matrice de ~20 000 à 500.
class TopKMultiLabelBinarizer:
    def __init__(self, top_k=500):
        self.top_k = top_k
        self.classes_ = None
    
    def fit_transform(self, X):
        # Compter les fréquences
        from collections import Counter
        counts = Counter([item for sublist in X for item in sublist])
        # Garder les top_k
        top_stars = {k for k, v in counts.most_common(self.top_k)}
        self.classes_ = sorted(list(top_stars))
        
        # Filtrer les listes
        X_filtered = [[item for item in sublist if item in top_stars] for sublist in X]
        
        # Utiliser le MLB standard sur les données filtrées
        self.mlb = MultiLabelBinarizer(classes=self.classes_)
        return self.mlb.fit_transform(X_filtered)

# Utilisation de la classe optimisée
mlb_stars = TopKMultiLabelBinarizer(top_k=500) 
stars_features = mlb_stars.fit_transform(movies_df['stars_list'])
n_stars = len(mlb_stars.classes_)
print(f"Processed {n_stars} unique stars (Top 500).")
# 3c. Languages
print("Processing Languages...")
movies_df['languages_list'] = movies_df['languages'].apply(parse_list_column)
movies_df['primary_language'] = movies_df['languages_list'].apply(lambda x: x[0] if len(x) > 0 else 'Unknown')
lang_encoder = LabelEncoder()
language_features = lang_encoder.fit_transform(movies_df['primary_language'])
language_features = language_features.reshape(-1, 1)
n_languages = len(lang_encoder.classes_)

# 3d. Numerical Features
print("Processing Numerical Features...")
numerical_cols = ['rating', 'votes', 'duration', 'méta_score']
for col in numerical_cols:
    movies_df[col] = pd.to_numeric(movies_df[col], errors='coerce')
    movies_df[col] = movies_df[col].fillna(0)

scaler = MinMaxScaler()
numerical_features = scaler.fit_transform(movies_df[numerical_cols])
n_numerical_features = numerical_features.shape[1]


# --- Step 4: Define the Content Tower Keras Model (CORRIGÉ) ---

print("\nBuilding Content Tower Keras model...")

CONTENT_EMBEDDING_SIZE = 32 

# --- Define Input Layers ---

# 1. SBERT Input
sbert_input = Input(shape=(SBERT_DIM,), name='sbert_embedding_input')

# 2. Numerical Input
numerical_input = Input(shape=(n_numerical_features,), name='numerical_input')

# 3. Genres Input
genre_input = Input(shape=(n_genres,), name='genre_input')

# 4. Stars Input (REMPLACE KEYWORDS)
# Utilisation de n_stars défini plus haut
stars_input = Input(shape=(n_stars,), name='stars_input') 

# 5. Language Input
language_input = Input(shape=(1,), name='language_input')


# --- Process Each Input Path ---

# 1. SBERT Path
sbert_vec = Dense(64, activation='relu', name='sbert_dense_1')(sbert_input)
sbert_vec = Dropout(0.2)(sbert_vec)

# 2. Numerical Path
numerical_vec = BatchNormalization(name='numerical_batchnorm')(numerical_input)

# 3. Genres Path
genre_vec = Dense(16, activation='relu', name='genre_dense')(genre_input)

# 4. Stars Path (REMPLACE KEYWORDS)
# On compresse fortement car le vecteur stars peut être très grand
stars_vec = Dense(32, activation='relu', name='stars_dense')(stars_input)

# 5. Language Path
lang_embedding = Embedding(input_dim=n_languages,
                           output_dim=8,
                           name='language_embedding')(language_input)
lang_vec = Flatten(name='language_flatten')(lang_embedding)


# --- Concatenate All Processed Vectors ---
print("Concatenating all feature vectors...")
concat = Concatenate(name='content_concat')([
    sbert_vec,
    numerical_vec,
    genre_vec,
    stars_vec, # On utilise stars_vec ici
    lang_vec
])

# --- Final Compression Layers ---
content_dense_1 = Dense(128, activation='relu', name='content_dense_1')(concat)
content_dense_1 = Dropout(0.3)(content_dense_1)
content_dense_2 = Dense(64, activation='relu', name='content_dense_2')(content_dense_1)

# Final Output
content_vector = Dense(CONTENT_EMBEDDING_SIZE, 
                       activation='linear',
                       name='content_vector_output')(content_dense_2)

# --- Create the Model Instance ---
# On met à jour la liste des inputs
content_tower_model = Model(
    inputs=[sbert_input, numerical_input, genre_input, stars_input, language_input],
    outputs=content_vector,
    name='Content_Tower'
)

# --- Show Model Summary ---
content_tower_model.summary()

Loading Data...
Loading existing embeddings from movie_embeddings.npy...
SBERT embedding dimension: 384
Starting feature preprocessing...
Processing Genres...
Processing Stars...
Processed 500 unique stars (Top 500).
Processing Languages...
Processing Numerical Features...

Building Content Tower Keras model...
Concatenating all feature vectors...


Model: "Content_Tower"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ sbert_embedding_in… │ (None, 384)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ language_input      │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sbert_dense_1       │ (None, 64)        │     24,640 │ sbert_embedding_… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ numerical_input     │ (None, 4)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ genre_input         │ (None, 192)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stars_input         │ (None, 500)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ language_embedding  │ (None, 1, 8)      │      1,136 │ language_input[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 64)        │          0 │ sbert_dense_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ numerical_batchnorm │ (None, 4)         │         16 │ numerical_input[… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ genre_dense (Dense) │ (None, 16)        │      3,088 │ genre_input[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stars_dense (Dense) │ (None, 32)        │     16,032 │ stars_input[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ language_flatten    │ (None, 8)         │          0 │ language_embeddi… │
│ (Flatten)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ content_concat      │ (None, 124)       │          0 │ dropout_2[0][0],  │
│ (Concatenate)       │                   │            │ numerical_batchn… │
│                     │                   │            │ genre_dense[0][0… │
│                     │                   │            │ stars_dense[0][0… │
│                     │                   │            │ language_flatten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ content_dense_1     │ (None, 128)       │     16,000 │ content_concat[0… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, 128)       │          0 │ content_dense_1[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ content_dense_2     │ (None, 64)        │      8,256 │ dropout_3[0][0]   │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ content_vector_out… │ (None, 32)        │      2,080 │ content_dense_2[… │
│ (Dense)             │                   │            │                 

 Total params: 71,248 (278.31 KB)

 Trainable params: 71,240 (278.28 KB)

 Non-trainable params: 8 (32.00 B)

In [26]:
# Block 3: Item Tower (Item-Based Collaborative Filtering)
# --------------------------------------------------------
# This script builds the Item Tower. It learns a "latent" embedding
# for each movie ID, capturing behavioral patterns (ratings) that
# content features alone cannot explain.

# --- Step 0: Imports ---
from sklearn.preprocessing import LabelEncoder

# Ensure we access the 'id' column
movie_ids = movies_df['id'].unique()
print(f"Found {len(movie_ids)} unique movie IDs.")
movie_ids = movies_df['id'].unique()


# --- Step 2: Encode Movie IDs ---
print("Encoding Movie IDs...")

movie_encoder = LabelEncoder()
movie_encoder.fit(movie_ids)
n_movies = len(movie_encoder.classes_)

# --- Step 3: Save the Encoder ---
# CRITICAL: We need this to map real movie IDs to the model later.
with open('movie_id_encoder.pkl', 'wb') as f:
    pickle.dump(movie_encoder, f)
print("Movie ID encoder saved to 'movie_id_encoder.pkl'.")


# --- Step 4: Define the Item Tower Keras Model ---

print("\nBuilding Item Tower Keras model...")

# Hyperparameter: Embedding Size
# Ideally, this should match the output size of the Content Tower (32)
# so they can be easily combined (added or concatenated).
ITEM_EMBEDDING_SIZE = 32 

# 1. Input Layer
# Expects a single integer (the movie index)
item_input = Input(shape=(1,), name='item_id_input')

# 2. Embedding Layer
item_embedding = Embedding(input_dim=n_movies,
                           output_dim=ITEM_EMBEDDING_SIZE,
                           name='item_embedding')(item_input)

# 3. Flatten Layer
item_vec = Flatten(name='item_flatten')(item_embedding)

# --- Create the Model Instance ---
item_tower_model = Model(
    inputs=item_input, 
    outputs=item_vec, 
    name='Item_Tower'
)

item_tower_model.summary()

print("\n--- Item Tower Ready ---")
print(f"Input: Movie Index (0 to {n_movies-1})")
print(f"Output: Latent Vector (Shape: {ITEM_EMBEDDING_SIZE})")

Found 63249 unique movie IDs.
Encoding Movie IDs...
Movie ID encoder saved to 'movie_id_encoder.pkl'.

Building Item Tower Keras model...


Model: "Item_Tower"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ item_id_input (InputLayer)      │ (None, 1)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ item_embedding (Embedding)      │ (None, 1, 32)          │     2,023,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ item_flatten (Flatten)          │ (None, 32)             │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,023,968 (7.72 MB)

 Trainable params: 2,023,968 (7.72 MB)

 Non-trainable params: 0 (0.00 B)


--- Item Tower Ready ---
Input: Movie Index (0 to 63248)
Output: Latent Vector (Shape: 32)


In [27]:
print("Noms des colonnes trouvés :", links_df.columns.tolist())
print("Aperçu des 2 premières lignes :")
print(links_df.head(2))

Noms des colonnes trouvés : ['movieId', 'imdbId', 'tmdbId']
Aperçu des 2 premières lignes :
   movieId  imdbId  tmdbId
0        1  114709   862.0
1        2  113497  8844.0


In [38]:
# Block 4: Dual-Tower Architecture (Fixed Data Loading)
# -----------------------------------------------------
# Architecture : Two-Tower (Dot Product)
# Data Loading : Version robuste utilisant 'links.csv' et 'imdb_tt_id'

import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.utils import Sequence
from tensorflow.keras.layers import Dense, Dropout, Concatenate, Input, Dot
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split

print("\n=== STARTING BLOCK 4: DUAL-TOWER TRAINING (ROBUST) ===")

# ==============================================================================
# STEP 0: CHARGEMENT ET TRADUCTION ROBUSTE (La partie critique)
# ==============================================================================
print("\n--- Step 0: Data Synchronization ---")

# 2. Nettoyage des colonnes LINKS
links_df.columns = links_df.columns.str.lower().str.strip()
if 'movieid' in links_df.columns: links_df.rename(columns={'movieid': 'movieId'}, inplace=True)

# 3. ANALYSE DE LA CIBLE (movies_df)
# On regarde quel format d'ID est utilisé par l'encodeur (Bloc 3)
target_id_example = movies_df['id'].iloc[0]
print(f"Format cible dans movies_df: {target_id_example} (Type: {type(target_id_example)})")

# Détection : Est-ce que ça ressemble à 'tt0114709' ?
is_imdb_format = isinstance(target_id_example, str) and target_id_example.startswith('tt')

# 4. CHOIX DE LA COLONNE DE LIEN
link_col = None
if is_imdb_format:
    print("Stratégie: Cible IMDB détectée (tt...).")
    if 'imdb_tt_id' in links_df.columns:
        link_col = 'imdb_tt_id' # Le cas idéal
    elif 'imdbid' in links_df.columns:
        link_col = 'imdbid' # Nécessitera un formatage
else:
    print("Stratégie: Cible TMDB/Numérique détectée.")
    if 'tmdbid' in links_df.columns: link_col = 'tmdbid'
    elif 'tmdb_id' in links_df.columns: link_col = 'tmdb_id'

if not link_col:
    raise ValueError("Impossible de trouver une colonne de lien compatible dans links.csv")

print(f"Colonne pivot choisie : {link_col}")

# 5. NETTOYAGE AVANT MERGE
# On s'assure que movieId est un entier partout
ratings_df['movieId'] = pd.to_numeric(ratings_df['movieId'], errors='coerce')
ratings_df = ratings_df.dropna(subset=['movieId'])
ratings_df['movieId'] = ratings_df['movieId'].astype(int)

links_df['movieId'] = pd.to_numeric(links_df['movieId'], errors='coerce')
links_df = links_df.dropna(subset=['movieId', link_col])
links_df['movieId'] = links_df['movieId'].astype(int)

# Si la colonne pivot est numérique (ex: tmdbid), on la force en int
if pd.api.types.is_numeric_dtype(links_df[link_col]):
    links_df[link_col] = links_df[link_col].astype(int)

# 6. LE MERGE
print("Exécution du merge...")
ratings_merged = pd.merge(ratings_df, links_df[['movieId', link_col]], on='movieId', how='inner')
print(f"Lignes après merge : {len(ratings_merged)}")

if len(ratings_merged) == 0:
    raise ValueError("Merge échoué (0 lignes). Vérifiez les types de données dans ratings.csv et links.csv")

# 7. FORMATAGE FINAL DES IDs
if is_imdb_format:
    # Si la cible est 'tt...'
    if link_col == 'imdb_tt_id':
        # C'est déjà bon
        ratings_merged['final_id'] = ratings_merged[link_col]
    else:
        # C'était imdbid (int), il faut formater
        ratings_merged['final_id'] = ratings_merged[link_col].apply(lambda x: 'tt{:07d}'.format(int(x)))
else:
    # Si la cible est numérique (ou string numérique)
    if isinstance(target_id_example, str):
        ratings_merged['final_id'] = ratings_merged[link_col].astype(str)
    else:
        ratings_merged['final_id'] = ratings_merged[link_col]

# Remplacement
ratings_df = ratings_merged.drop(columns=['movieId', link_col]).rename(columns={'final_id': 'movieId'})


# ==============================================================================
# STEP 1: FEATURE MAPPING
# ==============================================================================
print("\n--- Step 1: Mapping Features ---")
# Mise à jour encodage
movies_df['encoded_movie_id'] = movie_encoder.transform(movies_df['id'])

movie_feature_map = {}
indices = movies_df.index.tolist()
enc_ids = movies_df['encoded_movie_id'].tolist()

for i, idx in enumerate(indices):
    e_id = enc_ids[i]
    movie_feature_map[e_id] = {
        'sbert': sbert_features[i],
        'num': numerical_features[i],
        'genre': genres_features[i],
        'stars': stars_features[i],
        'lang': language_features[i]
    }

# ==============================================================================
# STEP 2: FILTERING & SPLIT
# ==============================================================================
print("\n--- Step 2: Filtering ---")
known_users = set(user_encoder.classes_)
known_movies = set(movie_encoder.classes_)

# Vérification visuelle
print(f"Exemple ID Rating: {ratings_df['movieId'].iloc[0]}")
print(f"Exemple ID Encoder: {list(known_movies)[0]}")

ratings_df = ratings_df[ratings_df['userId'].isin(known_users)]
ratings_df = ratings_df[ratings_df['movieId'].isin(known_movies)]

if len(ratings_df) == 0: 
    raise ValueError("0 Ratings après filtrage. Les IDs ne correspondent toujours pas.")
    
print(f"Ratings validés : {len(ratings_df)}")

ratings_df['user_encoded'] = user_encoder.transform(ratings_df['userId'])
ratings_df['movie_encoded'] = movie_encoder.transform(ratings_df['movieId'])

X_train, X_val, y_train, y_val = train_test_split(
    ratings_df[['user_encoded', 'movie_encoded']],
    ratings_df['rating'],
    test_size=0.2, 
    random_state=42
)

# ==============================================================================
# STEP 3: GENERATOR (Fixed for Dictionary Input)
# ==============================================================================
class DualTowerGenerator(Sequence):
    def __init__(self, user_ids, movie_ids, ratings, batch_size=256, shuffle=True):
        self.user_ids = user_ids
        self.movie_ids = movie_ids
        self.ratings = ratings
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.indices = np.arange(len(self.user_ids))
        if self.shuffle: np.random.shuffle(self.indices)

    def __len__(self):
        return int(np.ceil(len(self.user_ids) / self.batch_size))

    def __getitem__(self, index):
        start = index * self.batch_size
        end = (index + 1) * self.batch_size
        idx = self.indices[start:end]
        
        b_users = self.user_ids[idx]
        b_movies = self.movie_ids[idx]
        b_ratings = self.ratings[idx]
        
        b_sbert, b_num, b_genre, b_stars, b_lang = [], [], [], [], []
        for m in b_movies:
            f = movie_feature_map.get(m)
            if f:
                b_sbert.append(f['sbert'])
                b_num.append(f['num'])
                b_genre.append(f['genre'])
                b_stars.append(f['stars'])
                b_lang.append(f['lang'])
            else:
                b_sbert.append(np.zeros(384))
                b_num.append(np.zeros(n_numerical_features))
                b_genre.append(np.zeros(n_genres))
                b_stars.append(np.zeros(n_stars))
                b_lang.append([0])

        inputs = {
            'user_input': np.array(b_users),
            'item_input': np.array(b_movies),
            'sbert_input': np.array(b_sbert),
            'num_input': np.array(b_num),
            'genre_input': np.array(b_genre),
            'stars_input': np.array(b_stars),
            'lang_input': np.array(b_lang)
        }
        return inputs, np.array(b_ratings)

    def on_epoch_end(self):
        if self.shuffle: np.random.shuffle(self.indices)

train_gen = DualTowerGenerator(X_train['user_encoded'].values, X_train['movie_encoded'].values, y_train.values, batch_size=128)
val_gen = DualTowerGenerator(X_val['user_encoded'].values, X_val['movie_encoded'].values, y_val.values, batch_size=128, shuffle=False)

# ==============================================================================
# STEP 4: DUAL-TOWER MODEL DEFINITION
# ==============================================================================


print("\n--- Step 4: Building Dual-Tower Model ---")

LATENT_DIM = 32 

# --- Inputs ---
user_in  = Input(shape=(1,), name='user_input')
item_in  = Input(shape=(1,), name='item_input')
sbert_in = Input(shape=(SBERT_DIM,), name='sbert_input')
num_in   = Input(shape=(n_numerical_features,), name='num_input')
genre_in = Input(shape=(n_genres,), name='genre_input')
stars_in = Input(shape=(n_stars,), name='stars_input')
lang_in  = Input(shape=(1,), name='lang_input')

# --- MOVIE TOWER (Item + Content) ---
item_emb = item_tower_model(item_in)
content_raw = content_tower_model([sbert_in, num_in, genre_in, stars_in, lang_in])
content_proj = Dense(LATENT_DIM, activation='relu')(content_raw)

# Fusion
movie_concat = Concatenate()([item_emb, content_proj])
final_movie_vec = Dense(LATENT_DIM, activation='linear', name='final_movie_embedding')(movie_concat)

# --- USER TOWER ---
user_emb = user_tower_model(user_in)
final_user_vec = Dense(LATENT_DIM, activation='linear', name='final_user_embedding')(user_emb)

# --- INTERACTION (Dot Product) ---
dot = Dot(axes=1)([final_user_vec, final_movie_vec])
output = Dense(1, activation='linear', name='rating_prediction')(dot)

# --- Compile ---
dual_model = Model(
    inputs=[user_in, item_in, sbert_in, num_in, genre_in, stars_in, lang_in],
    outputs=output,
    name='Dual_Tower_Model'
)

dual_model.compile(optimizer=Adam(0.001), loss='mse', metrics=['mae'])
dual_model.summary()

# ==============================================================================
# STEP 5: TRAINING
# ==============================================================================
print("\n--- Step 5: Training ---")

history = dual_model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=8,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(patience=1, factor=0.2)
    ]
)

dual_model.save('dual_tower_model.h5')
print("Dual Tower Model Saved.")


=== STARTING BLOCK 4: DUAL-TOWER TRAINING (ROBUST) ===

--- Step 0: Data Synchronization ---
Format cible dans movies_df: tt0027483 (Type: <class 'str'>)
Stratégie: Cible IMDB détectée (tt...).
Colonne pivot choisie : imdbid
Exécution du merge...
Lignes après merge : 100836

--- Step 1: Mapping Features ---

--- Step 2: Filtering ---
Exemple ID Rating: tt0114709
Exemple ID Encoder: tt21431644
Ratings validés : 98714

--- Step 4: Building Dual-Tower Model ---


Model: "Dual_Tower_Model"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ sbert_input         │ (None, 384)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ num_input           │ (None, 4)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ genre_input         │ (None, 192)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stars_input         │ (None, 500)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lang_input          │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ item_input          │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Content_Tower       │ (None, 32)        │     71,248 │ sbert_input[0][0… │
│ (Functional)        │                   │            │ num_input[0][0],  │
│                     │                   │            │ genre_input[0][0… │
│                     │                   │            │ stars_input[0][0… │
│                     │                   │            │ lang_input[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ user_input          │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Item_Tower          │ (None, 32)        │  2,023,968 │ item_input[0][0]  │
│ (Functional)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 32)        │      1,056 │ Content_Tower[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ User_Tower          │ (None, 32)        │     19,520 │ user_input[0][0]  │
│ (Functional)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_1       │ (None, 64)        │          0 │ Item_Tower[0][0], │
│ (Concatenate)       │                   │            │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ final_user_embeddi… │ (None, 32)        │      1,056 │ User_Tower[0][0]  │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ final_movie_embedd… │ (None, 32)        │      2,080 │ concatenate_1[0]… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dot (Dot)           │ (None, 1)         │          0 │ final_user_embed… │
│                     │                   │            │ final_movie_embe… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rating_prediction   │ (None, 1)         │          2 │ dot[0][0]         │
│ (Dense)             │                   │            │                 

 Total params: 2,118,930 (8.08 MB)

 Trainable params: 2,118,922 (8.08 MB)

 Non-trainable params: 8 (32.00 B)


--- Step 5: Training ---
Epoch 1/8


/usr/local/lib/python3.11/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


617/617 ━━━━━━━━━━━━━━━━━━━━ 13s 11ms/step - loss: 2.5748 - mae: 1.1472 - val_loss: 0.7767 - val_mae: 0.6976 - learning_rate: 0.0010
Epoch 2/8
617/617 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - loss: 0.7052 - mae: 0.6455 - val_loss: 0.7184 - val_mae: 0.6625 - learning_rate: 0.0010
Epoch 3/8
617/617 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - loss: 0.6330 - mae: 0.6071 - val_loss: 0.7313 - val_mae: 0.6716 - learning_rate: 0.0010
Epoch 4/8
617/617 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - loss: 0.5330 - mae: 0.5533 - val_loss: 0.7225 - val_mae: 0.6594 - learning_rate: 2.0000e-04
Epoch 5/8
617/617 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - loss: 0.5029 - mae: 0.5352 - val_loss: 0.7326 - val_mae: 0.6662 - learning_rate: 4.0000e-05


Dual Tower Model Saved.


In [40]:
# Block 5: Item-to-Item Inference (Dual-Tower Architecture)
# ---------------------------------------------------------
# On extrait les vecteurs "Super-Film" (Taille 32) générés par la tour Item+Content.
# Ces vecteurs sont géométriquement alignés pour refléter le contenu ET les préférences.

import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from tensorflow.keras.models import Model, load_model

print("\n=== STARTING BLOCK 5: DUAL-TOWER INFERENCE ===")

# --- Step 1: Load Resources ---
print("Loading Dual-Tower Model...")

try:
    # On charge le modèle sauvegardé à la fin du Bloc 4
    model = load_model('dual_tower_model.h5')
    print("Model loaded successfully.")
except:
    print("Modèle introuvable. Utilisation du modèle en mémoire (si disponible).")
    if 'dual_model' not in locals():
        raise ValueError("Aucun modèle disponible. Veuillez exécuter le Bloc 4.")
    model = dual_model

# Vérification des features (doivent être en mémoire depuis le Bloc 2)
if 'sbert_features' not in locals():
    raise ValueError("CRITIQUE : Les variables du Bloc 2 (sbert_features, etc.) ne sont pas définies.")


# --- Step 2: Configure Vector Extractor ---
print("\n--- Configuring Movie Vector Extractor ---")

# CIBLE : On veut la sortie de la tour FILM, juste avant le Dot Product.
# Dans le Bloc 4, nous avons nommé cette couche : 'final_movie_embedding'
target_layer = 'final_movie_embedding'

try:
    movie_output = model.get_layer(target_layer).output
    print(f"Couche cible trouvée : '{target_layer}'")
    
    # INPUTS : Le modèle a 7 inputs.
    # L'input 0 est 'user_input'. Les inputs 1 à 6 sont pour le film.
    # On prend donc slice [1:]
    movie_inputs = model.input[1:]
    
    # Création du sous-modèle
    vector_model = Model(inputs=movie_inputs, outputs=movie_output)
    print("Extracteur créé. Input: Features du Film -> Output: Vecteur Latent (32)")

except ValueError as e:
    print(f"ERREUR: Impossible de trouver la couche '{target_layer}'.")
    print("Vérifiez le 'model.summary()' du Bloc 4.")
    raise e


# --- Step 3: Generate Vectors for ALL Movies ---
print("\n--- Generating Super-Vectors ---")

# 1. Alignement des IDs
# On doit s'assurer que l'ordre des films correspond à movies_df
# et que les IDs sont bien encodés avec le movie_encoder
all_tmdb_ids = movies_df['id'].values

# Petit hack de sécurité : si movies_df contient des films inconnus de l'encodeur
# (ce qui peut arriver si le filtrage ratings a été agressif), on les filtre.
try:
    encoded_ids = movie_encoder.transform(all_tmdb_ids)
    final_movies_df = movies_df # Tout va bien
    
    # Features correspondantes
    f_sbert = sbert_features
    f_num = numerical_features
    f_genre = genres_features
    f_stars = stars_features
    f_lang = language_features

except ValueError:
    print("Attention : Certains films de movies_df sont inconnus de l'encodeur.")
    print("Filtrage des films pour l'inférence...")
    
    mask = np.isin(all_tmdb_ids, movie_encoder.classes_)
    final_movies_df = movies_df[mask].copy()
    encoded_ids = movie_encoder.transform(final_movies_df['id'].values)
    
    # Filtrage des arrays numpy
    f_sbert = sbert_features[mask]
    f_num = numerical_features[mask]
    f_genre = genres_features[mask]
    f_stars = stars_features[mask]
    f_lang = language_features[mask]
    
    print(f"Films conservés pour recommandation : {len(final_movies_df)}")


# 2. Construction de la liste d'inputs pour l'inférence
# Ordre STRICT du modèle Dual-Tower :
# [Item(1), SBERT(2), Num(3), Genre(4), Stars(5), Lang(6)]
inference_inputs = [
    encoded_ids,
    f_sbert,
    f_num,
    f_genre,
    f_stars,
    f_lang
]

# 3. Prédiction
# On obtient une matrice de taille (N_FILMS, 32)
movie_vectors = vector_model.predict(inference_inputs, verbose=1)
print(f"Vecteurs générés. Shape : {movie_vectors.shape}")


# --- Step 4: Compute Similarity ---
print("\n--- Calculating Similarity Matrix ---")
# Avec des vecteurs de taille 32, c'est très rapide et léger en RAM
sim_matrix = cosine_similarity(movie_vectors)
print("Matrice de similarité calculée.")


# --- Step 5: Recommendation Function ---

# Indexation pour recherche rapide (Titre -> Position dans le DF filtré)
indices = pd.Series(np.arange(len(final_movies_df)), index=final_movies_df['title'])
# Gestion des doublons de titres (on garde le dernier)
indices = indices[~indices.index.duplicated(keep='last')]

def get_recommendations(title, N=10):
    """
    Recommande des films basés sur la similarité vectorielle Dual-Tower.
    """
    # 1. Trouver l'index
    try:
        idx = indices[title]
    except KeyError:
        print(f"Film '{title}' non trouvé dans la base active.")
        return

    # 2. Récupérer les scores
    # idx est déjà la position numérique (row index) grâce à np.arange
    sim_scores = list(enumerate(sim_matrix[idx]))
    
    # 3. Trier
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    
    # 4. Prendre les N meilleurs (en excluant le premier qui est lui-même)
    sim_scores = sim_scores[1:N+1]
    
    movie_indices = [i[0] for i in sim_scores]
    
    # 5. Affichage
    print(f"\n--- Recommandations pour '{title}' ---")
    for i, (m_idx, score) in enumerate(sim_scores):
        rec_title = final_movies_df['title'].iloc[m_idx]
        # On ajoute le genre pour vérifier la pertinence du Content Based
        rec_genres = final_movies_df['genres'].iloc[m_idx] if 'genres' in final_movies_df else "N/A"
        print(f"{i+1}. {rec_title} (Sim: {score:.3f})")


# --- Step 6: Final Testing ---
print("\n--- TEST PHASE ---")

test_list = ['The Matrix', 'Toy Story', 'The Godfather', 'Interstellar', 'Minions']

for t in test_list:
    if t in indices:
        get_recommendations(t)
    else:
        print(f"Skipping '{t}' (not in dataset)")


=== STARTING BLOCK 5: DUAL-TOWER INFERENCE ===
Loading Dual-Tower Model...
Modèle introuvable. Utilisation du modèle en mémoire (si disponible).

--- Configuring Movie Vector Extractor ---
Couche cible trouvée : 'final_movie_embedding'
Extracteur créé. Input: Features du Film -> Output: Vecteur Latent (32)

--- Generating Super-Vectors ---
1977/1977 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step
Vecteurs générés. Shape : (63249, 32)

--- Calculating Similarity Matrix ---
Matrice de similarité calculée.

--- TEST PHASE ---

--- Recommandations pour 'The Matrix' ---
1. Avengers: Endgame (Sim: 0.999)
2. The Internet's Own Boy: The Story of Aaron Swartz (Sim: 0.999)
3. The Threesome (Sim: 0.999)
4. Waste Land (Sim: 0.999)
5. The Way I See It (Sim: 0.999)
6. Hidden Figures (Sim: 0.999)
7. Waterloo Bridge (Sim: 0.999)
8. Walk the Line (Sim: 0.999)
9. The Last Lions (Sim: 0.999)
10. The Pursuit of Happyness (Sim: 0.999)

--- Recommandations pour 'Toy Story' ---
1. Monsters, Inc. (Sim: 0.999)
2. Paris Is B